# Notebook 03 — Validation, Drift & Adversarial Robustness
**Master Playbook Section 7, 9, 19 (Phase 3)**

Depends on Notebook 02 having produced `champion_model.pkl` and having
printed the chosen threshold. Set `CHOSEN_THRESHOLD` below to that real
value before running.

In [ ]:
import sys, os, pickle
sys.path.insert(0, os.path.abspath("../"))
import numpy as np
import pandas as pd

import two_gate_validation as tgv
import drift_monitoring as dm
import adversarial_robustness as ar

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
DATA_PATH = "creditcard.csv"
CHOSEN_THRESHOLD = None  # <-- set this to Notebook 02's real printed threshold before running

if CHOSEN_THRESHOLD is None:
    raise ValueError("Set CHOSEN_THRESHOLD to the real value Notebook 02 printed before running this notebook.")

df = pd.read_csv(DATA_PATH)
feature_cols = [c for c in df.columns if c not in ("Time", "Class")]

with open("champion_model.pkl", "rb") as f:
    champion_model = pickle.load(f)

## Gate 1 — structural integrity checks (Section 7)

In [ ]:
gate1 = tgv.run_gate1_structural_checks(df, feature_cols + ["Time"], "Class")
print(gate1)

## Gate 2 — concentration report & stress test (Section 7, 19)

In [ ]:
df["pred"] = (champion_model.predict_proba(df[feature_cols])[:, 1] >= CHOSEN_THRESHOLD).astype(int)

concentration = tgv.concentration_report_by_amount_and_time(
    df, amount_col="Amount", time_col="Time", y_true_col="Class", y_pred_col="pred", n_amount_bands=5
)
concentration

In [ ]:
BASE_FRAUD_RATE = df["Class"].mean()
stress = tgv.stress_test_scenario(
    df, amount_col="Amount", scenario_volume_multiplier=2.0, scenario_fraud_rate_multiplier=3.0,
    base_fraud_rate=BASE_FRAUD_RATE,
)
print(stress)

## Drift monitoring — real early/late window proxy (Section 9)
This dataset has no real elapsed-time history, so the disclosed 48-hour-window proxy splits the real Time column in half.

In [ ]:
early, late = dm.early_vs_late_window_proxy(df, "Time")
print(f"Early window: {len(early)} rows, fraud rate {early['Class'].mean():.4%}")
print(f"Late window:  {len(late)} rows, fraud rate {late['Class'].mean():.4%}")

top_feature = "V14"  # commonly high-SHAP-importance in this dataset — replace with YOUR real top-SHAP feature
psi = dm.population_stability_index(early[top_feature].values, late[top_feature].values)
print(f"PSI on {top_feature} (early vs late): {psi:.4f}")

## Adversarial robustness — perturbation sensitivity + boundary-search evasion (Section 19, Phase 3)
The real, honest test of how easily this exact model could be evaded by realistic amount/time manipulation.

In [ ]:
def score_fn(X_in):
    return champion_model.predict_proba(X_in[feature_cols])[:, 1]

X_fraud = df.loc[df["Class"] == 1].copy()

sens_results = ar.perturbation_sensitivity_test(score_fn, X_fraud, CHOSEN_THRESHOLD)
boundary_result = ar.boundary_search_attack(score_fn, X_fraud, CHOSEN_THRESHOLD)
ar.print_report(sens_results, boundary_result)

## Your real conclusions (fill in AFTER running)

- **Gate 1:** pass / fail — _[detail any failure]_
- **Concentration finding:** _[is fraud/error concentrated in a specific Amount band or hour?]_
- **PSI on the real early/late split:** _[value]_ — alert per Section 9's tier thresholds? _[yes/no]_
- **Adversarial evasion rate within budget:** _[value]_ — is this acceptable for production? _[your honest judgment, with reasoning]_
